In [ ]:
# pip install langchain langchain-openai langchain-community
# pip install azure-storage-blob azure-search-documents

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import AzureBlobStorageContainerLoader
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage

# ------------------------------------------------------------
# 1. AZURE OPENAI
# ------------------------------------------------------------

llm = AzureChatOpenAI(
    azure_endpoint="https://YOUR-RESOURCE.openai.azure.com/",
    api_key="YOUR_OPENAI_API_KEY",
    api_version="2024-10-21",
    azure_deployment="gpt-4o",          # use your actual deployment name
    temperature=0
)

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://YOUR-RESOURCE.openai.azure.com/",
    api_key="YOUR_OPENAI_API_KEY",
    api_version="2024-10-21",
    azure_deployment="text-embedding-3-large"
)

# ------------------------------------------------------------
# 2. LOAD DOCUMENTS FROM AZURE BLOB STORAGE
# ------------------------------------------------------------

loader = AzureBlobStorageContainerLoader(
    conn_str="YOUR_BLOB_CONNECTION_STRING",
    container="documents"
)

documents: list[Document] = loader.load()

# ------------------------------------------------------------
# 3. CHUNKING
# ------------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks: list[Document] = splitter.split_documents(documents)

# ------------------------------------------------------------
# 4 + 5. EMBED & STORE IN AZURE AI SEARCH
# (AzureSearch handles embedding + upsert in one call)
# ------------------------------------------------------------

# Prerequisites in Azure AI Search:
#   - Create an index (langchain auto-creates if it doesn't exist)
#   - Fields: id (Edm.String, key), content (Edm.String), 
#             content_vector (Collection(Edm.Single)), metadata_json (Edm.String)
#   - Enable vector search profile on content_vector

AZURE_SEARCH_ENDPOINT = "https://YOUR-SEARCH-SERVICE.search.windows.net"
AZURE_SEARCH_KEY      = "YOUR_SEARCH_ADMIN_KEY"
INDEX_NAME            = "rag-index"

vector_store = AzureSearch(
    azure_search_endpoint=AZURE_SEARCH_ENDPOINT,
    azure_search_key=AZURE_SEARCH_KEY,
    index_name=INDEX_NAME,
    embedding_function=embeddings.embed_query,
    # Optional: switch to hybrid (keyword + vector) search
    search_type="hybrid",               # "similarity" | "hybrid" | "semantic_hybrid"
)

# Upserts chunks with embeddings — skip if index already populated
vector_store.add_documents(documents=chunks)

# ------------------------------------------------------------
# 6 + 7. QUERY & RETRIEVE
# ------------------------------------------------------------

query = "What is the company's leave policy?"

retrieved_docs: list[Document] = vector_store.similarity_search(
    query=query,
    k=5,
    # For semantic_hybrid search, add:
    # semantic_configuration_name="my-semantic-config"
)

# ------------------------------------------------------------
# 8. BUILD CONTEXT
# ------------------------------------------------------------

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

# ------------------------------------------------------------
# 9. GENERATE ANSWER
# ------------------------------------------------------------

prompt = f"""Answer the question using ONLY the provided context.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say "I don't know based on the provided documents."
"""

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)